# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below we enumerate all available record sets and their fields. All references use their Croissant `@id` properties.

In [ ]:
# List all record sets and their fields by @id
record_sets = list(dataset.record_sets.values())
if not record_sets:
    print('No record sets found in the schema.')
else:
    for rs in record_sets:
        print(f"Record Set Name: {rs.name}")
        print(f"  @id: {rs.id}")
        print(f"  Fields in this Record Set:")
        for f in rs.fields.values():
            line = f"    - Field: {f.name} (@id: {f.id}; dataType: {f.data_type})"
            print(line)
        print("")

In [ ]:
# Optionally, enumerate a few sample records from each record set (by @id)
if record_sets:
    for rs in record_sets:
        print(f'First 2 records of record set {rs.name} (@id: {rs.id}):')
        try:
            for i, record in enumerate(dataset.records(record_set=rs.id)):
                print(record)
                if i >= 1:
                    break
        except Exception as e:
            print(f"  Could not load records: {e}")
        print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Here, we extract all record sets, load them into Pandas DataFrames, and explore one.

In [ ]:
# Extract all record sets into DataFrames, using their @id
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets.values()]
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records from record set @id={rs_id}")
    except Exception as e:
        print(f"Could not load records for record set {rs_id}: {e}")
        dataframes[rs_id] = pd.DataFrame()

# Choose a record set for further analysis
if record_set_ids:
    chosen_record_set_id = record_set_ids[0]
    print(f"\nColumns in record set @id={chosen_record_set_id}:")
    print(dataframes[chosen_record_set_id].columns.tolist())
    dataframes[chosen_record_set_id].head()
else:
    print('No record sets found in the dataset for extraction.')

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

> Edit the variables below to match a numeric field and a group field as needed (identifying by `@id` from overview above).

In [ ]:
import numpy as np

# Specify the record set and fields for EDA by @id
record_set_id = chosen_record_set_id if record_set_ids else None
numeric_field_id = None
group_field_id = None

if record_set_id:
    df = dataframes[record_set_id]
    # Attempt to automatically guess a numeric field by type or name
    sample_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if sample_fields:
        numeric_field_id = sample_fields[0] # Choose the first numeric column
    else:
        print("No numeric columns found for EDA.")
    # Attempt to find a group field (string/categorical)
    group_fields = df.select_dtypes(include=[object]).columns.tolist()
    if group_fields:
        group_field_id = group_fields[0]
    print(f"Using numeric field for EDA: {numeric_field_id}")
    print(f"Using group field: {group_field_id}")
    
    if numeric_field_id:
        # Remove NaNs
        filtered_df = df[df[numeric_field_id].notnull()]
        # Filter for values above the mean
        threshold = filtered_df[numeric_field_id].mean()
        filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > mean ({threshold:.2f}):")
        print(filtered_df.head(3))

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head(3))

        # Optionally, group by a categorical field and compute means
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by {group_field_id}:")
            print(grouped_df.head())
else:
    print('No record set selected for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Run the cell, and edit the variables to match suitable fields if needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(data=df, x=numeric_field_id, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in record set {record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot by group, if available
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric field available for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we demonstrated how to:
- Load and inspect a Croissant dataset schema with `mlcroissant`.
- Identify record sets and fields by Croissant `@id`.
- Extract tabular data for analysis.
- Apply basic EDA and visualize numeric fields.

You can review the dataset schema and field definitions for detailed analysis or tailor the workflow for further research questions.